<a href="https://colab.research.google.com/github/Aakashi06/First-AI-Agent-/blob/main/First_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install -U "smolagents[toolkit]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 8.2 MB/s eta 0:00:00


In [13]:
import smolagents

print("smolagents version:", smolagents.__version__)

smolagents version: 1.26.0


In [19]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF token loaded:", HF_TOKEN is not None)

HF token loaded: True


In [20]:
from huggingface_hub import HfApi

api = HfApi(token=HF_TOKEN)

user = api.whoami()

print("Authenticated as:", user["name"])

Authenticated as: Aakashi06


Test the model



In [26]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    token=HF_TOKEN
)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.1-8B-Instruct",
    messages=[
        {
            "role": "user",
            "content": "What is an AI agent? Explain it in 4  simple points."
        }
    ],
    max_tokens=200
)

print(response.choices[0].message.content)

Here are 4 simple points explaining what an AI agent is:

**1. Definition**: An AI agent is a computer program that can perform tasks autonomously, making decisions and taking actions based on its own programming and algorithms.

**2. Autonomy**: AI agents can operate independently, without human intervention, to accomplish specific goals or tasks. They can interact with their environment, receive feedback, and adapt to new situations.

**3. Decision-making**: AI agents use machine learning algorithms and data analysis to make decisions, often based on complex data and rules. They can learn from experience, recognize patterns, and improve their performance over time.

**4. Interactions**: AI agents can interact with humans and other systems, such as sensors, devices, or other AI systems, to achieve their goals. They can communicate through various interfaces, like voice, text, or visual displays, and can even learn to understand and respond to natural language inputs.


Create an actual agent

In [27]:
from smolagents import (
    CodeAgent,
    InferenceClientModel,
    DuckDuckGoSearchTool,
    tool
)

In [28]:
model = InferenceClientModel(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    token=HF_TOKEN,
    max_tokens=2048,
    temperature=0.5
)

Create agent

Start with no tools

In [29]:
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=5
)

Test no tool agent

In [30]:
result = agent.run(
    "Calculate the sum of all numbers from 1 to 10."
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Calculate the sum of all numbers from 1 to 10.                                                                  │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  numbers = list(range(1, 11))  # Generate a list of numbers from 1 to 10                                          
  print(numbers)  # Print the list of numbers                                                                      
  total_sum = sum(numbers)  # Calculate the sum of the numbers                                                     
  final_answer(total_sum)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Final answer: 55

[Step 1: Duration 7.69 seconds| Input tokens: 1,999 | Output tokens: 99]

55


In [31]:
search_tool = DuckDuckGoSearchTool()

Create agent with tool

In [32]:
agent = CodeAgent(
    tools=[search_tool],
    model=model,
    max_steps=6
)

Test Web Search

In [36]:
result = agent.run(
    "Search the web and tell me what Hugging Face is "
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search the web and tell me what Hugging Face is                                                                 │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  hugging_face_description = web_search(query="Hugging Face")                                                      
  print(hugging_face_description)                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Hugging Face](https://en.wikipedia.org/wiki/Hugging_Face)
Hugging Face, Inc., is an American company based in New York City that develops computation tools for building 
applications using machine learning. Hugging Face's transformers library is built for natural language processing 
applications. The Hugging Face platform allows users to share machine learning models and datasets and showcase 
their work.

[Models – Hugging Face](https://huggingface.co/models)
4 days ago - Hugging Face · Main · Tasks · Libraries · Languages · Licenses · Other · Tasks · Text Generation · 
Any-to-Any · Image-Text-to-Text · Image-to-Text · Image-to-Image · Text-to-Image · Text-to-Video · Text-to-Speech +
44 · Parameters Reset Parameters ·

[What is Hugging Face? - IBM](https://www.ibm.com/think/topics/hugging-face)
What is Hugging Face? Hugging Face is a company that maintains a huge open-source community of the same name that 
builds tools, machine learning models and platforms for working with artificial intelligence, with a focus on data 
science, machine learning and natural language processing (NLP).

[Hugging Face](https://grokipedia.com/page/Hugging_Face)
Hugging Face is an American-French artificial intelligence company and open-source platform that facilitates 
collaboration in machine learning, particularly through its Hugging Face...

[Hugging Face Tutorial - 
GeeksforGeeks](https://www.geeksforgeeks.org/artificial-intelligence/hugging-face-tutorial/)
Apr 9, 2026 · Hugging Face is an open-source AI platform that provides pre-trained models, datasets and tools to 
build Natural Language Processing, computer vision, audio and generative AI applications. It helps developers to 
use AI models without training everything from scratch.

[New details in the OpenAI Hugging Face hack show how far 
...](https://www.cnbc.com/2026/07/30/open-ai-hugging-face-hack-latest.html)
Jul 30, 2026 · Industry experts, researchers and government officials have been rattled by the Hugging Face 
incident, and many expressed their concern on social media in recent days.

[OpenAI and Hugging Face partner to address security incident during model evaluation | 
OpenAI](https://openai.com/index/hugging-face-model-evaluation-security-incident/)
3 weeks ago - OpenAI and Hugging Face share early findings from a security incident during AI model evaluation, 
highlighting advanced cyber capabilities and lessons for defenders.

[Hugging Face | OpenAI](https://openai.com/business/plugins/hugging-face/)
3 days ago - Connect to the Hugging Face Hub in ChatGPT to explore models, datasets, and metadata and inspect 
options without manual browsing.

[What is Hugging Face? The 2026 Guide to the AI Hub | 
metacto](https://www.metacto.com/blogs/what-is-hugging-face-a-guide-to-the-ai-community-and-its-tools)
May 31, 2026 - Hugging Face is the open-source platform and community where the world’s AI models, datasets, and 
demos live. It is best understood as “GitHub for machine learning”—a hub where researchers, companies, and indie 
developers publish, discover, ...

[What Is Hugging Face? | Coursera](https://www.coursera.org/articles/what-is-hugging-face)
May 30, 2026 - You can upload ML models to Hugging Face for tasks like image classification and processing, text 
summarization, translation, and question answering. Hugging Face can also help understand and categorize emotions 
in text into predefined labels.

Out: None

[Step 1: Duration 2.91 seconds| Input tokens: 2,048 | Output tokens: 47]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  hugging_face_description = "Hugging Face is an American company based in New York City that develops             
  computation tools for building applications using machine learning. It provides a platform for users to share    
  machine learning models and datasets, and showcases their work. Hugging Face's transformers library is built     
  for natural language processing applications."                                                                   
  final_answer(hugging_face_description)                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Hugging Face is an American company based in New York City that develops computation tools for 
building applications using machine learning. It provides a platform for users to share machine learning models and
datasets, and showcases their work. Hugging Face's transformers library is built for natural language processing 
applications.

[Step 2: Duration 3.13 seconds| Input tokens: 4,970 | Output tokens: 138]

Hugging Face is an American company based in New York City that develops computation tools for building applications using machine learning. It provides a platform for users to share machine learning models and datasets, and showcases their work. Hugging Face's transformers library is built for natural language processing applications.


Make tool


In [37]:
@tool
def calculate_percentage(value:float, percentage:float) -> float:

   """
    Calculate a percentage of a number.

    Args:
        value: The original number.
        percentage: The percentage to calculate.

    Returns:
        The calculated percentage.
    """
   return value * percentage/100

In [38]:
agent = CodeAgent(
    tools=[
        search_tool,
        calculate_percentage
    ],
    model=model,
    max_steps=6
)

In [40]:
result = agent.run(
    """
    Search the web for India's current population in 2026.

    From the search results, identify ONE reliable numerical
    population figure.

    Then calculate exactly 5% of that population.

    Do not use the search-result text itself as the number.
    Extract the actual numerical population first.

    Give me:
    1. The population figure you selected
    2. The calculation
    3. The final answer
    """
)

print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search the web for India's current population in 2026.                                                          │
│                                                                                                                 │
│     From the search results, identify ONE reliable numerical                                                    │
│     population figure.                                                                                          │
│                                                                                                                 │
│     Then calculate exactly 5% of that population.                                                               │
│                                                                                                                 │
│     Do not use the search-result text itself as the number.                                                     │
│     Extract the actual numerical population first.                                                              │
│                                                                                                                 │
│     Give me:                                                                                                    │
│     1. The population figure you selected                                                                       │
│     2. The calculation                                                                                          │
│     3. The final answer                                                                                         │
│                                                                                                                 │
╰─ InferenceClientModel - meta-llama/Llama-3.1-8B-Instruct ───────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  search_results = web_search(query="India current population 2026")                                               
  print(search_results)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[India Population (2026) - Worldometer](https://www.worldometers.info/world-population/india-population/)
Notes. The India Population (Live) counter shows a continuously updated estimate of the current population of India
delivered by Worldometer's RTS algorithm, which processes data collected from the United Nations Population 
Division. The Population of India (1950 - 2026)...

[India Population 2026](https://worldpopulationreview.com/countries/india)
As of 2026, India has a total population of 1,476,625,576, ranking as the 1st most populous nation in the world. 
India has a net population change of approximately 34,660 people per day, driven by 62,939 births, 27,073 deaths, 
and net emigration of 1,206 people.

[India population (2026) live — Countrymeters](https://countrymeters.info/en/India)
The current population of India is 1,491,330,662 as of Wednesday, August 05, 2026. Population clock live, current, 
historical and projected population.

[India Population (2026)](https://populationtoday.com/in-india/)
Discover the latest population statistics for India and explore demographic trends with our interactive data 
visualizations.Population of India today. 1,477,751,312. Tuesday, August 4, 2026.

[India Population Pyramid (1950–2100) | PopulationPyramid.net](https://www.populationpyramid.net/india/)
A population pyramid shows the age and sex distribution of a population. India in 2026: total population 
1,476,625,576. Median age: 30.2 years.

[India Population in 2026- Current Population of 
India](https://www.indiaconvey.com/population/india-current-population.html)
India’s Population is projected to grow by 36,811 people every day in 2026. Explore details on current population 
of India which is estimated to be 1,466,298,116 (1.46 billion).

[India Population, 1951 – 2026 | CEIC Data](https://www.ceicdata.com/en/indicator/india/population)
Key information about India population. The India population reached 1,420.6 million people in Mar 2026, compared 
with the previously reported figure of 1,408.1 million people in Mar 2025.

[World Population 2026 | Total Population by Country, Trends...](https://www.populationreview.org/world)
The current world population is 8.04 billion (2026 estimate). Total population by country, regional breakdowns, 
growth rate, and UN projections to 2050 and 2100.

[World population live, August 2026](https://nationsgeo.com/population/)
Live world population, calculate world population based on changes in world births and deaths for all world 
countries.Thursday, August 06, 2026. Sources V.27 Edition UN Population Division. The current world population of 
2026.

[India Population Statistics 2026 | Current Population of 
India](https://www.theglobalstatistics.com/india-population-statistics/)
According to the data report, Population of India in 2026 is estimated to be 1,428,000,000 with a growth rate of 
1.2%. The population of India doubled to 1.2 billion between 1975 and 2010 and reaching the billion mark in 1998.

Out: None

[Step 1: Duration 6.07 seconds| Input tokens: 2,169 | Output tokens: 85]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 5 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The population figure selected is 1,476,625,576. The calculation is 5% of the population, which    
  is 73,831,278.8.")                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The population figure selected is 1,476,625,576. The calculation is 5% of the population, which is 
73,831,278.8.

[Step 5: Duration 2.54 seconds| Input tokens: 17,244 | Output tokens: 385]

The population figure selected is 1,476,625,576. The calculation is 5% of the population, which is 73,831,278.8.
